In [61]:
import numpy as np
import pandas as pd
import re
import string

In [62]:
text = 'great product. i love that'

In [63]:
def remove_punctuations(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation, '')
    return text

In [64]:
with open('../static/model/corpora/stopwords/english', 'r') as file:
    stop_words = file.read().splitlines()

In [65]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [66]:
def preprocessing(text):
    data = pd.DataFrame([text],columns=['tweet'])
    data['tweet'] = data['tweet'].apply(lambda x: " ".join(x.lower () for x in x.split()))
    data['tweet'] = data['tweet'].apply(lambda x: " ".join(re.sub(r'^https?:\/\/.*[\r\n]*', '',x,flags=re.MULTILINE) for x in x.split()))
    data['tweet'] = data['tweet'].str.replace('\d+', '', regex = True)
    data['tweet'] = data['tweet'].apply(remove_punctuations)
    data['tweet'] = data['tweet'].apply(lambda x: " ".join (x for x in x.split() if x not in stop_words))
    data['tweet'] = data['tweet'].apply(lambda x: " ".join(ps.stem(x) for x in x.split()))
    return data['tweet']

In [67]:
preprocessed_txt = preprocessing(text)
preprocessed_txt

0    great product love
Name: tweet, dtype: object

In [68]:
vocab = pd.read_csv('../static/model/vocabulary.text', header=None)
tokens = vocab[0].tolist()

In [69]:
def vectorizer(data_set, vocabulary):
    vectorized_list = []

    for sentence in data_set:
        sentence_list = np.zeros(len(vocabulary))

        for i in range (len(vocabulary)):
            if vocabulary[i] in sentence.split():
                sentence_list[i] = 1
        vectorized_list.append(sentence_list)
    vectorized_list_new = np.asarray(vectorized_list, dtype=np.float32)

    return vectorized_list_new

In [70]:
vectorized_txt = vectorizer(preprocessed_txt,tokens)
vectorized_txt

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 1145), dtype=float32)

In [71]:
#load the model into pipiline
import pickle
with open('../static/model/model.pickle', 'rb') as f:
    model = pickle.load(f)

ModuleNotFoundError: No module named 'sklearn'